In [ ]:
##DailyChallenge

# ============================================================================
# RÉGRESSION LOGISTIQUE - PRÉDICTION D'ADMISSION À L'UNIVERSITÉ
# ============================================================================

# IMPORT DES BIBLIOTHÈQUES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# ============================================================================
# 1. EXPLORATION DES DONNÉES
# ============================================================================

print("="*60)
print("1. EXPLORATION DES DONNÉES")
print("="*60)

# Chargement des données
# Note: Le fichier n'a pas d'en-tête, nous devons donc nommer les colonnes
data = pd.read_csv('ex2data1.csv', header=None, names=['Exam1', 'Exam2', 'Admitted'])

# Affichage des premières lignes
print("\n Aperçu des données (5 premières lignes) :")
print(data.head())

# Informations sur le dataset
print("\n Informations sur les données :")
print(data.info())

# Statistiques descriptives
print("\n Statistiques descriptives :")
print(data.describe())

# Vérification des valeurs manquantes
print(f"\n Valeurs manquantes : {data.isnull().sum().sum()}")

# Distribution de la variable cible
admitted_counts = data['Admitted'].value_counts()
print(f"\n Distribution des admissions :")
print(f"   Non admis (0) : {admitted_counts[0]} étudiants")
print(f"   Admis (1)     : {admitted_counts[1]} étudiants")
print(f"   Total         : {len(data)} étudiants")

# ============================================================================
# 2. VISUALISATION DES DONNÉES (NUAGE DE POINTS)
# ============================================================================

print("\n" + "="*60)
print("2. VISUALISATION DES DONNÉES")
print("="*60)

# Création du nuage de points
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1: Nuage de points simple
ax1.scatter(data[data['Admitted'] == 0]['Exam1'], 
           data[data['Admitted'] == 0]['Exam2'], 
           color='red', alpha=0.7, label='Non admis (0)', s=50)
ax1.scatter(data[data['Admitted'] == 1]['Exam1'], 
           data[data['Admitted'] == 1]['Exam2'], 
           color='green', alpha=0.7, label='Admis (1)', s=50)
ax1.set_xlabel('Note Examen 1', fontsize=12)
ax1.set_ylabel('Note Examen 2', fontsize=12)
ax1.set_title('Résultats aux examens par statut d\'admission', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Graphique 2: Distribution des notes par examen
ax2.hist([data[data['Admitted'] == 0]['Exam1'], 
          data[data['Admitted'] == 1]['Exam1']], 
         bins=15, alpha=0.7, label=['Non admis', 'Admis'], color=['red', 'green'])
ax2.set_xlabel('Note Examen 1', fontsize=12)
ax2.set_ylabel('Nombre d\'étudiants', fontsize=12)
ax2.set_title('Distribution des notes - Examen 1', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Graphique supplémentaire : Distribution des notes Examen 2
fig2, ax3 = plt.subplots(figsize=(10, 5))
ax3.hist([data[data['Admitted'] == 0]['Exam2'], 
          data[data['Admitted'] == 1]['Exam2']], 
         bins=15, alpha=0.7, label=['Non admis', 'Admis'], color=['red', 'green'])
ax3.set_xlabel('Note Examen 2', fontsize=12)
ax3.set_ylabel('Nombre d\'étudiants', fontsize=12)
ax3.set_title('Distribution des notes - Examen 2', fontsize=14)
ax3.legend()
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n Analyse visuelle :")
print("   - Les étudiants admis (verts) semblent avoir de meilleures notes aux deux examens")
print("   - Il existe une séparation partielle entre les deux groupes")
print("   - La régression logistique devrait pouvoir trouver une frontière de décision")

# ============================================================================
# 3. APPLICATION DE LA RÉGRESSION LOGISTIQUE
# ============================================================================

print("\n" + "="*60)
print("3. RÉGRESSION LOGISTIQUE AVEC SCIKIT-LEARN")
print("="*60)

# Séparation des features (X) et de la cible (y)
X = data[['Exam1', 'Exam2']]
y = data['Admitted']

# Division en ensembles d'entraînement et de test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\n Taille de l'ensemble d'entraînement : {len(X_train)}")
print(f" Taille de l'ensemble de test : {len(X_test)}")

# Création et entraînement du modèle de régression logistique
# On augmente max_iter pour assurer la convergence
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

# Affichage des paramètres du modèle
print(f"\n Paramètres du modèle :")
print(f"   Coefficients : {model.coef_[0]}")
print(f"   Intercept : {model.intercept_[0]:.4f}")

# Équation de la frontière de décision
coef1, coef2 = model.coef_[0]
intercept = model.intercept_[0]
print(f"\n Équation de la frontière de décision :")
print(f"   {coef1:.4f} * Exam1 + {coef2:.4f} * Exam2 + {intercept:.4f} = 0")
print(f"   Soit : Exam2 = -({coef1:.4f} * Exam1 + {intercept:.4f}) / {coef2:.4f}")

# ============================================================================
# 4. PRÉDICTIONS ET ÉVALUATION DU MODÈLE
# ============================================================================

print("\n" + "="*60)
print("4. PRÉDICTIONS ET ÉVALUATION")
print("="*60)

# Prédictions sur l'ensemble de test
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calcul de la précision (accuracy)
accuracy = accuracy_score(y_test, y_pred)
print(f"\n📊 Précision du modèle (Accuracy) : {accuracy:.4f} ({accuracy*100:.2f}%)")

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
print(f"\n📊 Matrice de confusion :")
print(f"   Vrais Négatifs (TN) : {cm[0,0]}")
print(f"   Faux Positifs (FP)  : {cm[0,1]}")
print(f"   Faux Négatifs (FN)  : {cm[1,0]}")
print(f"   Vrais Positifs (TP) : {cm[1,1]}")

# Rapport de classification détaillé
print(f"\n📊 Rapport de classification :")
print(classification_report(y_test, y_pred, target_names=['Non admis', 'Admis']))

# ============================================================================
# 5. VISUALISATION DE LA FRONTIÈRE DE DÉCISION
# ============================================================================

print("\n" + "="*60)
print("5. VISUALISATION DE LA FRONTIÈRE DE DÉCISION")
print("="*60)

# Création d'une grille pour visualiser la frontière de décision
x_min, x_max = X['Exam1'].min() - 5, X['Exam1'].max() + 5
y_min, y_max = X['Exam2'].min() - 5, X['Exam2'].max() + 5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))

# Prédiction sur toute la grille
Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Visualisation
fig3, ax4 = plt.subplots(figsize=(12, 8))

# Affichage de la frontière de décision
ax4.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
ax4.contour(xx, yy, Z, colors='black', linewidths=1, levels=[0.5])

# Affichage des points de données
ax4.scatter(X[y == 0]['Exam1'], X[y == 0]['Exam2'], 
           color='red', alpha=0.7, label='Non admis', s=50, edgecolor='black')
ax4.scatter(X[y == 1]['Exam1'], X[y == 1]['Exam2'], 
           color='green', alpha=0.7, label='Admis', s=50, edgecolor='black')

# Tracé de la frontière de décision linéaire
x_line = np.linspace(x_min, x_max, 100)
y_line = -(coef1 * x_line + intercept) / coef2
ax4.plot(x_line, y_line, 'b-', linewidth=2, label='Frontière de décision')

ax4.set_xlabel('Note Examen 1', fontsize=12)
ax4.set_ylabel('Note Examen 2', fontsize=12)
ax4.set_title('Régression Logistique - Frontière de Décision', fontsize=14)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================================
# 6. COURBE ROC ET AUC
# ============================================================================

from sklearn.metrics import roc_curve, roc_auc_score

# Calcul de la courbe ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)

# Visualisation
fig4, ax5 = plt.subplots(figsize=(8, 6))
ax5.plot(fpr, tpr, 'b-', linewidth=2, label=f'Courbe ROC (AUC = {auc:.3f})')
ax5.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Classificateur aléatoire (AUC = 0.5)')
ax5.set_xlabel('Taux de Faux Positifs (1 - Spécificité)', fontsize=12)
ax5.set_ylabel('Taux de Vrais Positifs (Sensibilité)', fontsize=12)
ax5.set_title('Courbe ROC du Modèle de Régression Logistique', fontsize=14)
ax5.legend(loc='lower right')
ax5.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n AUC (Area Under the Curve) : {auc:.4f}")
print("   Interprétation de l'AUC :")
print("   - AUC = 1.0 : Modèle parfait")
print("   - AUC = 0.8-0.9 : Excellent")
print("   - AUC = 0.7-0.8 : Bon")
print("   - AUC = 0.6-0.7 : Acceptable")
print("   - AUC = 0.5   : Aléatoire")

# ============================================================================
# 7. ANALYSE DES ERREURS ET INTERPRÉTATION FINALE
# ============================================================================

print("\n" + "="*60)
print("6. ANALYSE DES RÉSULTATS ET INTERPRÉTATION")
print("="*60)

# Analyse des faux positifs et faux négatifs
fp_indices = np.where((y_test == 0) & (y_pred == 1))[0]
fn_indices = np.where((y_test == 1) & (y_pred == 0))[0]

print(f"\n Analyse des erreurs :")
print(f"   Nombre de faux positifs (étudiants non admis prédits comme admis) : {len(fp_indices)}")
print(f"   Nombre de faux négatifs (étudiants admis prédits comme non admis) : {len(fn_indices)}")

if len(fp_indices) > 0:
    print("\n   Étudiants en erreur (faux positifs) :")
    for idx in fp_indices[:5]:  # Affiche les 5 premiers
        print(f"   - Exam1={X_test.iloc[idx]['Exam1']:.2f}, Exam2={X_test.iloc[idx]['Exam2']:.2f}")

# Interprétation des coefficients
print("\n Interprétation des coefficients :")
print(f"   Coefficient Examen 1 : {coef1:.4f}")
print(f"   Coefficient Examen 2 : {coef2:.4f}")
print("\n   Interprétation :")
print("   - Un coefficient positif indique qu'une note plus élevée augmente la probabilité d'admission")
print("   - Un coefficient négatif indique qu'une note plus élevée diminue la probabilité d'admission")
print(f"   - Ici, les deux coefficients sont positifs : de meilleures notes aux deux examens")
print(f"     augmentent la probabilité d'être admis.")

print("\n" + "="*60)
print("CONCLUSION FINALE")
print("="*60)

print(f"""
 RÉSUMÉ DES RÉSULTATS :

1. QUALITÉ DU MODÈLE :
   - Précision (Accuracy) : {accuracy:.2%}
   - AUC : {auc:.3f}

2. PERFORMANCES PAR CLASSE :
   - Sensibilité (rappel) pour les admis : {cm[1,1]/(cm[1,0]+cm[1,1]):.2%}
   - Spécificité pour les non-admis : {cm[0,0]/(cm[0,0]+cm[0,1]):.2%}

3. INTERPRÉTATION :
   {'✓ Le modèle a de très bonnes performances' if accuracy > 0.85 else '⚠️ Le modèle a des performances moyennes'}
   {'✓ La frontière de décision sépare bien les deux classes' if auc > 0.85 else '⚠️ Il y a un chevauchement important entre les classes'}

4. RECOMMANDATIONS :
   - Un étudiant avec des notes élevées aux deux examens a une forte probabilité d'être admis
   - La frontière de décision peut être utilisée comme référence pour les admissions futures
   - Pour améliorer le modèle, on pourrait ajouter plus de features (expérience, lettres de recommandation, etc.)
""")